# AEC Pipeline — Kiểm chứng Acoustic Echo Cancellation

**Kịch bản thực tế:**
```
Người A (far-end) ──(mạng)──▶ Loa máy B ──(âm thanh dội tường)──▶ Mic máy B
```
Mic máy B thu được: **echo tiếng A** + **tiếng người B (near-end)**.  
AEC Pipeline khử echo → chỉ gửi lại tiếng người B cho A.

**Notebook này kiểm chứng 2 trường hợp:**
1. **Echo-only** (không có near-end) → output phải là **im lặng**
2. **Echo + Double-Talk** (có near-end beep) → output phải **chỉ còn tiếng beep**

Sử dụng đúng class `AECPipeline` — giống hệt Desktop App.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import sounddevice as sd
from scipy.signal import convolve
from scipy.io.wavfile import read, write
from IPython.display import Audio, display

from core.aec_pipeline import AECPipeline, AECConfig

# ═══════════════════════════════════════════════════════════════
# 1. Far-end signal (tiếng người A phát qua loa máy B)
#    True = ghi âm mới, False = dùng bản cũ (voice.wav)
# ═══════════════════════════════════════════════════════════════
RECORD_NEW = False          # ◄◄◄ Đổi thành True để ghi âm mới
VOICE_PATH = os.path.join("..", "voice.wav")
fs = 16000
seconds = 5

if RECORD_NEW:
    print(f"🎤 Đang ghi âm {seconds}s... Hãy nói gì đó!")
    far_end = sd.rec(int(seconds * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()
    far_end = far_end.flatten()
    far_end = far_end / (np.max(np.abs(far_end)) + 1e-10)
    write(VOICE_PATH, fs, (far_end * 32767).astype(np.int16))
    print(f"💾 Đã lưu {VOICE_PATH}")
else:
    fs_r, far_end = read(VOICE_PATH)
    far_end = far_end.astype(np.float32)
    if far_end.max() > 1.5:
        far_end = far_end / 32768.0
    fs = int(fs_r)
    print(f"📂 Đọc far-end từ voice.wav: {len(far_end)/fs:.1f}s, fs={fs} Hz")

# ═══════════════════════════════════════════════════════════════
# 2. Room Impulse Response (mô phỏng phòng có tường dội âm)
# ═══════════════════════════════════════════════════════════════
REFLECTIONS = [
    (0,    0.6),    # Direct path: loa → mic
    (50,   0.3),    # Tường 1 phản xạ sau 50ms
    (120,  0.15),   # Tường 2 phản xạ sau 120ms
    (250,  0.05),   # Tường 3 phản xạ sau 250ms
]

rir = np.zeros(int(500 / 1000 * fs))
for delay_ms, gain in REFLECTIONS:
    idx = int(delay_ms / 1000 * fs)
    if idx < len(rir):
        rir[idx] = gain

# Echo = far-end convolved with RIR
echo = convolve(far_end, rir, mode='full')[:len(far_end)]

# ═══════════════════════════════════════════════════════════════
# 3. Near-end signal (tiếng người B — mô phỏng bằng beep 1kHz)
# ═══════════════════════════════════════════════════════════════
DT_START, DT_END = 2.0, 2.5  # giây
near_end = np.zeros_like(far_end)
dt_s, dt_e = int(DT_START * fs), int(DT_END * fs)
t_dt = np.arange(dt_e - dt_s) / fs
near_end[dt_s:dt_e] = 0.5 * np.sin(2 * np.pi * 1000 * t_dt)

# ═══════════════════════════════════════════════════════════════
# 4. AEC Config — auto-tính filter_length từ RIR
# ═══════════════════════════════════════════════════════════════
FRAME_SIZE = 1024
max_rir_samples = max(int(d / 1000 * fs) for d, _ in REFLECTIONS)
filter_length = max(max_rir_samples + 256, 512)

# Biên độ chuẩn hóa: tất cả Audio player dùng cùng scale để so sánh chính xác
# Lấy max peak từ tín hiệu to nhất (echo+near_end) để không bao giờ vượt 1.0
AMP_REF = float(np.max(np.abs(echo + near_end))) + 0.01

print(f"RIR: {len(REFLECTIONS)} reflections, dài nhất = {max_rir_samples} samples ({max_rir_samples*1000/fs:.0f}ms)")
print(f"Filter length = {filter_length} taps ({filter_length*1000/fs:.0f}ms)")
print(f"Near-end (beep 1kHz): {DT_START}–{DT_END}s, gain=0.5")

📂 Đọc far-end từ voice.wav: 5.0s, fs=16000 Hz
RIR: 4 reflections, dài nhất = 4000 samples (250ms)
Filter length = 4256 taps (266ms)
Near-end (beep 1kHz): 2.0–2.5s, gain=0.5


## Test Case 1: Echo-only (không có near-end)

**Mic thu được:** chỉ có echo từ far-end (không có người B nói).  
**Kỳ vọng:** AEC khử hoàn toàn echo → output = **im lặng**.

```
Far-end → Loa → Phòng (RIR) → Mic = echo
AEC Pipeline(mic, far-end) → ~0 (silence)
```

In [5]:
# ═══════════════════════════════════════════════════════════════
# TEST 1: Echo-only → output phải im lặng
# ═══════════════════════════════════════════════════════════════
mic_echo_only = echo.copy()

config1 = AECConfig(
    sample_rate=fs,
    frame_size=FRAME_SIZE,
    filter_length=filter_length,
    mu=0.7,              # Aggressive cho offline test (hội tụ nhanh)
    nls_alpha=4.0,       # Over-subtraction mạnh (Wiener gain, không rè)
    nls_beta=0.0001,     # Gain floor = -80dB → NLS suppress thêm ≤80dB
)
pipeline1 = AECPipeline(config1)

num_frames = len(far_end) // FRAME_SIZE
output1 = np.zeros_like(far_end)

for i in range(num_frames):
    s = i * FRAME_SIZE
    e = s + FRAME_SIZE
    output1[s:e] = pipeline1.process(mic_echo_only[s:e], far_end[s:e])

# ── Số liệu thực tế ──
N = num_frames * FRAME_SIZE
mic_rms = np.sqrt(np.mean(mic_echo_only[:N].astype(np.float64)**2))
out_rms = np.sqrt(np.mean(output1[:N].astype(np.float64)**2))
erle1 = 10 * np.log10(mic_rms**2 / (out_rms**2 + 1e-10))

print("TEST 1 — Echo-only:")
print(f"  Mic input  RMS = {mic_rms:.6f}  (peak = {np.max(np.abs(mic_echo_only[:N])):.4f})")
print(f"  AEC output RMS = {out_rms:.6f}  (peak = {np.max(np.abs(output1[:N])):.4f})")
print(f"  ERLE = {erle1:.1f} dB")
print(f"  Output nhỏ hơn input {mic_rms/out_rms:.0f}x (RMS) | {np.max(np.abs(mic_echo_only[:N]))/(np.max(np.abs(output1[:N]))+1e-10):.0f}x (peak)")
print()

# Per-second breakdown
print("  Per-second breakdown (giây đầu = convergence, sau đó = steady-state):")
for sec in range(int(len(far_end)/fs)):
    s1, e1 = sec*fs, min((sec+1)*fs, N)
    if s1 >= N: break
    m_rms = np.sqrt(np.mean(mic_echo_only[s1:e1].astype(np.float64)**2))
    o_rms = np.sqrt(np.mean(output1[s1:e1].astype(np.float64)**2))
    erle_s = 10*np.log10(m_rms**2/(o_rms**2+1e-10))
    print(f"    Giây {sec}–{sec+1}: ERLE={erle_s:.1f}dB, suppress={m_rms/(o_rms+1e-10):.0f}x")

# Steady-state ERLE (sau convergence period ~1s)
ss_start = fs  # Bỏ giây đầu tiên (convergence)
ss_mic = np.sqrt(np.mean(mic_echo_only[ss_start:N].astype(np.float64)**2))
ss_out = np.sqrt(np.mean(output1[ss_start:N].astype(np.float64)**2))
ss_erle = 10*np.log10(ss_mic**2/(ss_out**2+1e-10))
print(f"\n  Overall ERLE = {erle1:.1f} dB ({mic_rms/out_rms:.0f}x)")
print(f"  Steady-state ERLE (giây 1–5) = {ss_erle:.1f} dB ({ss_mic/(ss_out+1e-10):.0f}x)")
print(f"  {'✅ PASS' if erle1 >= 40 else '❌ FAIL'}: Overall ERLE={erle1:.1f}dB ≥ 40dB")

TEST 1 — Echo-only:
  Mic input  RMS = 0.037296  (peak = 0.5990)
  AEC output RMS = 0.000008  (peak = 0.0002)
  ERLE = 69.2 dB
  Output nhỏ hơn input 4582x (RMS) | 3439x (peak)

  Per-second breakdown (giây đầu = convergence, sau đó = steady-state):
    Giây 0–1: ERLE=68.8dB, suppress=3153x
    Giây 1–2: ERLE=72.0dB, suppress=60195x
    Giây 2–3: ERLE=63.3dB, suppress=17805x
    Giây 3–4: ERLE=67.2dB, suppress=250283x
    Giây 4–5: ERLE=71.3dB, suppress=85047x

  Overall ERLE = 69.2 dB (4582x)
  Steady-state ERLE (giây 1–5) = 69.6 dB (52813x)
  ✅ PASS: Overall ERLE=69.2dB ≥ 40dB


In [6]:
# ⚠️ QUAN TRỌNG: normalize=False để giữ nguyên biên độ thật.
# Nếu normalize=True (mặc định), Audio sẽ khuếch đại tín hiệu nhỏ lên max volume
# → residual bé tí (RMS=0.0006) bị phóng to thành tiếng to → nghe tưởng chưa khử echo!
#
# Dùng AMP_REF để scale tất cả audio về cùng mức → so sánh độ to chính xác.

print("▶ Phát MIC INPUT (echo tiếng người A — nghe rõ):")
display(Audio(mic_echo_only / AMP_REF, rate=fs, normalize=False))

print("\n▶ Phát AEC OUTPUT (phải gần như im lặng):")
display(Audio(output1 / AMP_REF, rate=fs, normalize=False))

print(f"\n📊 So sánh: output nhỏ hơn input {np.max(np.abs(mic_echo_only))/np.max(np.abs(output1)):.0f}x về biên độ")

▶ Phát MIC INPUT (echo tiếng người A — nghe rõ):



▶ Phát AEC OUTPUT (phải gần như im lặng):



📊 So sánh: output nhỏ hơn input 3439x về biên độ


## Test Case 2: Echo + Double-Talk (có near-end beep)

**Mic thu được:** echo từ far-end + tiếng beep 1kHz (mô phỏng người B nói) ở giây 2.0–2.5.  
**Kỳ vọng:** AEC khử echo nhưng **giữ nguyên tiếng beep** → output chỉ còn beep.

```
Far-end → Loa → Phòng (RIR) → Mic = echo + near-end(beep)
AEC Pipeline(mic, far-end) → beep only
```

In [7]:
# ═══════════════════════════════════════════════════════════════
# TEST 2: Echo + Double-Talk → output phải chỉ còn tiếng beep
# ═══════════════════════════════════════════════════════════════
mic_with_dt = echo + near_end

config2 = AECConfig(
    sample_rate=fs,
    frame_size=FRAME_SIZE,
    filter_length=filter_length,
    mu=0.7,              # Aggressive cho offline test
    nls_alpha=4.0,       # Over-subtraction mạnh (Wiener gain)
    nls_beta=0.0001,     # Gain floor = -80dB
)
pipeline2 = AECPipeline(config2)

output2 = np.zeros_like(far_end)
for i in range(num_frames):
    s = i * FRAME_SIZE
    e = s + FRAME_SIZE
    output2[s:e] = pipeline2.process(mic_with_dt[s:e], far_end[s:e])

# ── Số liệu thực tế ──
N = num_frames * FRAME_SIZE

# Vùng echo-only (0 → DT_START): phải im lặng
echo_mic_rms = np.sqrt(np.mean(mic_with_dt[:dt_s].astype(np.float64)**2))
echo_out_rms = np.sqrt(np.mean(output2[:dt_s].astype(np.float64)**2))
echo_sup = 10 * np.log10(echo_mic_rms**2 / (echo_out_rms**2 + 1e-10))

# Vùng DT (DT_START → DT_END): beep phải còn nguyên
ne_rms = np.sqrt(np.mean(near_end[dt_s:dt_e].astype(np.float64)**2))
ne_out_rms = np.sqrt(np.mean(output2[dt_s:dt_e].astype(np.float64)**2))
ne_distortion = 10 * np.log10(ne_out_rms**2 / (ne_rms**2 + 1e-10))

# Vùng sau DT (DT_END → end): echo phải được khử tiếp
post_dt_s, post_dt_e = dt_e, N
post_mic_rms = np.sqrt(np.mean(mic_with_dt[post_dt_s:post_dt_e].astype(np.float64)**2))
post_out_rms = np.sqrt(np.mean(output2[post_dt_s:post_dt_e].astype(np.float64)**2))
post_sup = 10 * np.log10(post_mic_rms**2 / (post_out_rms**2 + 1e-10))

print("TEST 2 — Echo + Double-Talk:")
print()
print(f"  [0–{DT_START}s] Vùng echo-only (trước beep):")
print(f"    Mic RMS = {echo_mic_rms:.6f}, Output RMS = {echo_out_rms:.6f}")
print(f"    Echo suppression = {echo_sup:.1f} dB → output nhỏ hơn {echo_mic_rms/(echo_out_rms+1e-10):.0f}x")
print()
print(f"  [{DT_START}–{DT_END}s] Vùng double-talk (beep):")
print(f"    Near-end RMS = {ne_rms:.6f}, Output RMS = {ne_out_rms:.6f}")
print(f"    Distortion = {ne_distortion:.1f} dB (>-6dB = OK)")
print()
print(f"  [{DT_END}–end] Vùng echo-only (sau beep):")
print(f"    Mic RMS = {post_mic_rms:.6f}, Output RMS = {post_out_rms:.6f}")
print(f"    Echo suppression = {post_sup:.1f} dB")
print()
print(f"  {'✅ PASS' if echo_sup >= 30 else '❌ FAIL'}: Echo khử = {echo_sup:.1f}dB")
print(f"  {'✅ PASS' if ne_distortion > -6 else '❌ FAIL'}: Beep distortion = {ne_distortion:.1f}dB")

TEST 2 — Echo + Double-Talk:

  [0–2.0s] Vùng echo-only (trước beep):
    Mic RMS = 0.049324, Output RMS = 0.000243
    Echo suppression = 46.1 dB → output nhỏ hơn 203x

  [2.0–2.5s] Vùng double-talk (beep):
    Near-end RMS = 0.353553, Output RMS = 0.353550
    Distortion = -0.0 dB (>-6dB = OK)

  [2.5–end] Vùng echo-only (sau beep):
    Mic RMS = 0.028792, Output RMS = 0.000096
    Echo suppression = 49.5 dB

  ✅ PASS: Echo khử = 46.1dB
  ✅ PASS: Beep distortion = -0.0dB


In [8]:
# normalize=False + cùng AMP_REF → so sánh trung thực giữa input và output

print("▶ Phát MIC INPUT (echo + beep chen vào giây 2–2.5):")
display(Audio(mic_with_dt / AMP_REF, rate=fs, normalize=False))

print("\n▶ Phát AEC OUTPUT (chỉ còn tiếng beep, echo đã bị khử):")
display(Audio(output2 / AMP_REF, rate=fs, normalize=False))

print("\n▶ Phát NEAR-END GỐC (beep gốc để so sánh):")
display(Audio(near_end / AMP_REF, rate=fs, normalize=False))

▶ Phát MIC INPUT (echo + beep chen vào giây 2–2.5):



▶ Phát AEC OUTPUT (chỉ còn tiếng beep, echo đã bị khử):



▶ Phát NEAR-END GỐC (beep gốc để so sánh):
